# Insurance Behavior Lab — EDA sampai model

Data **sintetis**, bukan fakta bisnis. Jalankan sel berurutan. Notebook memanggil modul yang sama
dengan pipeline CLI sehingga preprocessing dan evaluasi tetap konsisten.
Sasaran: memahami tiga misteri bisnis, membangun prediksi, dan merancang eksperimen.
Output notebook disimpan di `artifacts/notebook_run/` agar hasil utama tetap mudah dibandingkan.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'insurance_lab').exists():
    ROOT = ROOT.parent
assert (ROOT / 'insurance_lab').exists(), 'Buka notebook dari folder proyek atau notebooks/'
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from insurance_lab.config import TARGET, FEATURES, split_months
from insurance_lab.data import (simulate_policies, engineer_policy_features,
    simulate_acquisition, engineer_acquisition_features, audit_policy_data)
from insurance_lab.modeling import train_churn, train_acquisition
OUT = ROOT / 'artifacts/notebook_run'
for folder in ['artifacts/tables', 'artifacts/models', 'data/processed', 'reports/figures']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
SEED = 26


## 1. Rumuskan masalah sebelum melihat model

Tulis dua hipotesis untuk setiap fenomena: premi naik tanpa churn langsung, churn melonjak
beberapa waktu kemudian, dan akuisisi meningkat di luar campaign.
Definisikan unit, horizon, outcome dan data tambahan yang diperlukan.
Jangan membuka formula simulator sebelum latihan hipotesis selesai.


In [ ]:
raw = simulate_policies(n_customers=6000, seed=SEED)
df = engineer_policy_features(raw)
audit = audit_policy_data(df)
display(pd.Series({k: v for k, v in audit.items() if k != 'missing_by_column'}))
display(df.head())


## 2. Audit data dan waktu

Apa arti `feature_as_of`, `snapshot_date`, dan `label_end_date`?
Mengapa satu nasabah muncul berulang? Bolehkah ia muncul sesudah terminal churn?
Missingness diisi **di dalam pipeline yang fit pada train**, bukan di seluruh DataFrame.


In [ ]:
display(df.isna().mean().sort_values(ascending=False).head(8))
display(df[FEATURES].describe(include='all').T)
assert (df.feature_as_of < df.snapshot_date).all()
assert (df.label_end_date > df.snapshot_date).all()


## 3. EDA: rate, denominator, segmen dan cohort

Bandingkan churn rate dan jumlah churn. Apakah perubahan jumlah pelanggan aktif mengubah
interpretasi? Hipotesis mana yang sesuai dengan pola renewal dan riwayat pembayaran?


In [ ]:
monthly = df.groupby('snapshot_date').agg(
    active=(TARGET, 'size'), churned=(TARGET, 'sum'), churn_rate=(TARGET, 'mean'))
display(monthly.tail(12))
monthly.churn_rate.mul(100).plot(figsize=(12, 4), title='Churn bulanan (%) | sintetis');


In [ ]:
segment = df.assign(near_renewal=df.months_to_renewal.le(1),
                    payment_issue=df.late_payments_3m.gt(0))
display(segment.groupby(['near_renewal', 'payment_issue'])[TARGET].agg(['size', 'mean']))


## 4. Fitur dan pemisahan waktu

Horizon adalah satu bulan kalender. Jul 2025 dan Jan 2026 adalah embargo.
Nasabah yang sama dapat berada di beberapa split pada waktu berbeda: ini meniru scoring
portofolio. Generalisasi ke nasabah yang benar-benar baru memerlukan evaluasi tambahan.


In [ ]:
splits = split_months(df)
display(pd.DataFrame([{'split': name, 'rows': len(part),
    'start': part.snapshot_date.min(), 'end': part.snapshot_date.max(),
    'prevalence': part[TARGET].mean()} for name, part in splits.items()]))
assert splits['train'].label_end_date.max() < splits['validation'].snapshot_date.min()
assert splits['validation'].label_end_date.max() < splits['test'].snapshot_date.min()
display(df[['premium_idr', 'income_monthly_idr', 'premium_to_income',
            'months_to_renewal', 'late_payments_3m', 'renewal_payment_interaction']].head())


## 5. Baseline dan model churn

Sel ini menjalankan tiga rolling backtest, fit train, pemilihan AP validation, final test,
permutation importance validation dan bootstrap customer. Model tidak dipilih dari test.
Perhatikan bahwa `average_precision` bukan trapezoidal PR-AUC.


In [ ]:
churn = train_churn(df, OUT, SEED)
print('Model terpilih:', churn['selected_model'])
display(churn['metrics'].round(4))
display(pd.read_csv(OUT / 'artifacts/tables/rolling_backtest.csv').round(4))


## 6. Error analysis dan kalibrasi

Top 10% dipilih per bulan. Confusion matrix memakai threshold tetap validation.
Mengapa hasil kedua aturan berbeda? Reliability diagram bersifat diagnostik;
recalibration belum diterapkan. Brier yang kecil saja belum menjamin kalibrasi baik.


In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import ConfusionMatrixDisplay
scored = churn['scored']
CalibrationDisplay.from_predictions(scored[TARGET], scored.score, n_bins=8, strategy='quantile');
ConfusionMatrixDisplay(churn['confusion']).plot();
display(churn['importance'].head(10))
display(scored.loc[scored.above_validation_threshold & scored[TARGET].eq(0)].head(3))
display(scored.loc[~scored.above_validation_threshold & scored[TARGET].eq(1)].head(3))


## 7. Akuisisi mingguan: jalur yang berbeda

Dataset akuisisi ini simulasi analitis terpisah dari daftar nasabah.
Lag hanya memakai minggu yang sudah terjadi. Semua wilayah dalam minggu sama masuk
split yang sama. Ini prediksi satu minggu ke depan pada setiap origin, bukan multi-step.


In [ ]:
acquisition_raw = simulate_acquisition(SEED)
acquisition_features = engineer_acquisition_features(acquisition_raw)
acquisition = train_acquisition(acquisition_features, OUT)
display(acquisition['metrics'].round(3))
display(acquisition['scored'].loc[acquisition['scored'].unusual_residual].head())


## 8. Buat seluruh visual dan baca dashboard

Periksa cohort retention: sel kosong adalah follow-up belum lengkap. Periksa grafik
sensitivitas: keuntungan bergantung pada asumsi rescue fraction yang belum diukur.


In [ ]:
from insurance_lab.visuals import create_figures
monthly, figures = create_figures(df, acquisition_raw, acquisition_features,
                                 churn, acquisition, OUT)
display(Image(filename=str(OUT / 'reports/figures/executive_dashboard.png')))
display(Image(filename=str(OUT / 'reports/figures/cohort_retention.png')))


## 9. Kesimpulan dan keputusan

Isi memo empat paragraf: fakta deskriptif; kualitas prediksi; hal yang belum diketahui;
eksperimen berikutnya. Jangan menyebut importance sebagai sebab atau angka simulasi
sebagai fakta Indonesia. Bandingkan kapasitas kontak 5%, 10%, 20% dengan skenario biaya.


In [ ]:
display(pd.read_csv(OUT / 'artifacts/tables/retention_sensitivity.csv'))
display(pd.DataFrame(churn['intervals']))


## 10. Uji inference dari artifact

Contoh ini memakai snapshot historis sintetis, bukan nasabah live. Gunakan joblib
hanya dari sumber tepercaya.


In [ ]:
import joblib
bundle = joblib.load(OUT / 'artifacts/models/churn_model.joblib')
example = pd.read_csv(OUT / 'data/processed/scoring_example.csv')
example['churn_probability'] = bundle['pipeline'].predict_proba(example[bundle['features']])[:, 1]
display(example[['customer_id', 'snapshot_date', 'churn_probability']].head(10))
